# Explore Streaming — Core Concepts with Native Spark

> **Module 2 · Part 1** | ~30 min | Level 300

In this notebook you will go hands-on with **Spark Structured Streaming** using plain PySpark — no frameworks, no abstractions. By the end you will have built a working **raw → bronze → silver** streaming pipeline and understand the primitives that every production streaming system is built on.

**Prerequisites** — The `stream_bronze_and_silver` Spark Job Definition should already be running. Data is landing in `Files/landing/` and being archived to `Files/archive/`.

### How Data Arrives

| Entity | Landing Format | Path / Endpoint |
|--------|---------------|----------------|
| item | JSON | `Files/landing/item/` |
| order | JSON | `Files/landing/order/` |
| shipment | JSON | `Files/landing/shipment/` |
| customer | Parquet | `Files/landing/customer/` |
| route, facility, servicelevel, exceptiontype | Parquet | `Files/landing/<name>/` |
| **shipment_scan_event** | **[Eventstream](https://learn.microsoft.com/fabric/real-time-intelligence/event-streams/overview)** ([Kafka protocol](https://learn.microsoft.com/fabric/real-time-intelligence/event-streams/add-source-custom-app#kafka)) | Fabric Eventstream endpoint |

JSON files follow an envelope pattern: `{"_meta": {...}, "data": [...]}`. Parquet files are flat with `OrganizationId` and `GeneratedAt` columns added by the data generator.

> ℹ️ _In production, high-velocity events like shipment scan events stream through [Fabric Eventstream](https://learn.microsoft.com/fabric/real-time-intelligence/event-streams/overview). For lab reproducibility we also provide a file-based fallback in `Files/archive/shipment_scan_event/`._

---

## 1 — Why Structured Streaming for a Medallion Pipeline?

[Structured Streaming](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) is not just for "real-time" — it is the engine of choice for **incremental batch** pipelines too. Two properties make it ideal for medallion architectures:

| Property | What it means |
|----------|---------------|
| **Checkpoint state** | Spark remembers which files (or offsets) it has already processed. Re-run the same job and it picks up only *new* data — no custom bookmarks required. |
| [**`availableNow` trigger**](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#triggers) | Process everything that has accumulated since the last run, then stop. You get **batch scheduling semantics** with **streaming state management**. |

> Think of `availableNow` as a batch job that *also* remembers where it left off.

### Side-by-Side: Batch Read vs. Streaming Read

The code is nearly identical — the only difference is `read` → `readStream` and the addition of a **checkpoint** + **trigger**.

Execute the cell below to read all customer parquet files in one batch:

In [ ]:
# --------- Batch ---------
# Reads all customer parquet files at once (full scan every time)
customer_batch_df = spark.read.parquet(
    'Files/archive/customer',
    recursiveFileLookup=True
)

print(f'Batch row count: {customer_batch_df.count()}')
display(customer_batch_df.limit(5))

Now execute the streaming version. Notice the three additions: **schema**, **checkpoint**, and **trigger**:

In [ ]:
# --------- Streaming (availableNow) ---------
# Reads only NEW customer parquet files since the last checkpoint
customer_stream_df = spark.readStream.schema(
    customer_batch_df.schema          # streaming requires an explicit schema
).parquet(
    'Files/archive/customer',
    recursiveFileLookup=True
)

query = (customer_stream_df.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', 'Files/explore/checkpoints/customer')
    .trigger(availableNow=True)
    .toTable('explore.customer')
)

query.awaitTermination()
print('Stream completed — only new files since the last checkpoint were processed.')

> 🔄 **Try it:** Run the streaming cell **a second time**. The stream finishes almost instantly because there is no new data — the checkpoint tracked what was already consumed.

---

## 2 — Write Streaming Queries on Top of Files

Now let's read **JSON** files. The shipment data follows a common envelope pattern from event stores and APIs:

```json
{
  "_meta": { "schemaVersion": "1.0", "producer": "lakegen.mcmillan", "recordType": "shipment", "enqueuedTime": "..." },
  "data": [ { "ShipmentId": "...", "TrackingNumber": "...", ... } ]
}
```

Execute the cell below to infer the schema from existing files.

> 💡 In production you should define schemas explicitly — inference reads all files and can be slow. See [Spark schema specification](https://spark.apache.org/docs/latest/sql-data-sources-json.html#data-source-option).

In [ ]:
shipment_schema = spark.read.json(
    'Files/archive/shipment',
    multiLine=True,
    recursiveFileLookup=True
).schema

print('Inferred schema — in production, define this explicitly.')
shipment_schema

Now stream the shipment files to a Delta table and wait for completion:

In [ ]:
shipment_stream_df = spark.readStream.schema(shipment_schema).json(
    'Files/archive/shipment',
    multiLine=True,
    recursiveFileLookup=True
)

shipment_query = (shipment_stream_df.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', 'Files/explore/checkpoints/shipment_raw')
    .trigger(availableNow=True)
    .toTable('explore.shipment_raw')
)

shipment_query.awaitTermination()
print('Shipment stream completed.')

### 📊 Observing a Stream

Every [`StreamingQuery`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#managing-streaming-queries) exposes two properties you will use constantly:

| Property | Purpose |
|----------|---------|
| `query.status` | Is the stream running, processing data, or stopped? |
| `query.lastProgress` | Metrics from the most recent micro-batch — input rows/sec, processed rows/sec, batch duration. |

Run the cell below to inspect the stream you just ran:

In [ ]:
print('=== status ===')
print(shipment_query.status)

print('\n=== lastProgress (key fields) ===')
lp = shipment_query.lastProgress
if lp:
    print(f'  batchId          : {lp["batchId"]}')
    print(f'  numInputRows     : {lp["numInputRows"]}')
    print(f'  inputRowsPerSec  : {lp["inputRowsPerSecond"]}')
    print(f'  processedRowsSec : {lp["processedRowsPerSecond"]}')
    print(f'  batchDuration    : {lp["batchDuration"]}')

### 🎯 Challenge: Query the Raw Shipment Table

Write a SQL query against the `explore.shipment_raw` table you just created. Use `EXPLODE(data)` to break the `data` array into individual rows and then expand the struct fields with `.*`.

Try it in the cell below!

In [ ]:
%%sql


<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal the answer</summary>

<br/>

**Approach:**
1. Inner query: `EXPLODE(data)` converts the array into rows with alias `shipment`
2. Outer query: `shipment.*` expands all struct fields into columns

```sql
SELECT shipment.*
FROM (
    SELECT EXPLODE(data) AS shipment
    FROM explore.shipment_raw
)
LIMIT 10;
```

**Key Takeaway:** This two-step pattern (explode → expand) is fundamental for flattening nested JSON in data engineering pipelines.

</details>

---

## 3 — Intro to the Streaming API

This section covers the key building blocks you combine in every streaming pipeline. See the [Spark Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) for the full reference.

### `read` vs `readStream`

| | `spark.read` | `spark.readStream` |
|---|---|---|
| Returns | Static `DataFrame` | Streaming `DataFrame` |
| Schema | Optional (can infer) | **Required** |
| Data scope | All data at call time | Only *new* data per trigger |

### Triggers

The **trigger** controls *when* and *how often* a micro-batch executes:

| Trigger | Behaviour |
|---------|----------|
| [`trigger(availableNow=True)`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#triggers) | Process all available data, then **stop**. Best for scheduled batch-style runs. |
| `trigger(processingTime='30 seconds')` | Continuously run a micro-batch every 30 s. |
| `trigger(once=True)` | Legacy — same intent as `availableNow` but processes in a single batch (less efficient). |
| *(no trigger)* | Continuous micro-batches as fast as possible. |

### `awaitTermination`

`query.awaitTermination()` **blocks** the calling cell until the stream finishes. Use it when:
- You run `availableNow` and want downstream cells to wait
- You orchestrate multiple sequential streams in one notebook

Without it, the stream runs **asynchronously** and the next cell executes immediately.

### Fan-Out with `foreachBatch`

[`foreachBatch`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#foreachbatch) gives you a **regular DataFrame** for each micro-batch, letting you write to **multiple sinks** or apply arbitrary logic:

```python
def write_to_multiple_tables(batch_df, batch_id):
    batch_df.write.format('delta').mode('append').saveAsTable('bronze.events')
    alerts = batch_df.where("severity = 'CRITICAL'")
    alerts.write.format('delta').mode('append').saveAsTable('bronze.alerts')

query = (stream_df.writeStream
    .foreachBatch(write_to_multiple_tables)
    .option('checkpointLocation', '...')
    .trigger(availableNow=True)
    .start()
)
```

> `foreachBatch` turns each micro-batch into a plain batch operation where any DataFrame API call works.

---

## 4 — Build a Raw → Bronze Streaming Pipeline

A good bronze layer applies three things to every source:

| Step | What it does |
|------|--------------|
| **Schema enforcement** | Parse the raw payload with a defined schema — no surprises downstream |
| **Column normalization** | Convert `PascalCase` to `snake_case` for consistency |
| **Audit columns** | Add `_processing_timestamp` so you know when each row was ingested |

### 🧪 The Development Challenge

> How do you develop against a live stream without writing garbage to your Delta tables?

**Answer:** Use the [**`memory` sink**](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#output-sinks). It writes micro-batch output to an in-memory temporary view — no checkpoints, no Delta writes, no cleanup.

Let's peek at the raw order JSON first:

In [ ]:
# Peek at raw order JSON — notice the _meta + data[] envelope
raw_order = spark.read.json(
    'Files/archive/order',
    multiLine=True,
    recursiveFileLookup=True
).limit(2)
display(raw_order)

### Stream a Sample to Memory

Execute the cell below to stream order data into an in-memory view — no Delta writes, no checkpoint:

In [ ]:
order_file_schema = raw_order.schema

mem_query = (spark.readStream
    .schema(order_file_schema)
    .json('Files/archive/order', multiLine=True, recursiveFileLookup=True)
    .writeStream
    .format('memory')
    .queryName('order_preview')
    .trigger(availableNow=True)
    .start()
)
mem_query.awaitTermination()

display(spark.sql('SELECT * FROM order_preview LIMIT 5'))

> The `memory` sink is your best friend during development. You get a queryable view without touching storage.

### Eventstream Source Pattern (Kafka)

In production, `shipment_scan_event` streams through **Fabric Eventstream** using the Kafka protocol. The read pattern looks like this:

```python
kafka_df = (spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', '<eventstream-endpoint>:9093')
    .option('kafka.sasl.mechanism', 'PLAIN')
    .option('kafka.security.protocol', 'SASL_SSL')
    .option('kafka.sasl.jaas.config',
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
            'required username="$ConnectionString" '
            'password="<connection-string>";')
    .option('subscribe', '<topic>')
    .option('startingOffsets', 'earliest')
    .load()
)

# The value column is a binary JSON payload — parse the _meta + data[] envelope
parsed = kafka_df.select(
    from_json(col('value').cast('string'), sse_envelope_schema).alias('payload')
).select('payload.*')
```

The same `memory` sink works identically for Eventstream sources — stream to memory, query the view, iterate.

> For this lab we use the **file-based fallback** so exercises work without a live Eventstream. The transforms are identical — only the reader changes. See the [Fabric Eventstream Kafka connector docs](https://learn.microsoft.com/fabric/real-time-intelligence/event-streams/add-source-custom-app#kafka) for production setup.

### Define Bronze Transforms

Run the cell below to define helper functions we will reuse for every source:

In [ ]:
import re
from pyspark.sql.functions import current_timestamp

def to_snake_case(name):
    """Convert PascalCase or camelCase to snake_case."""
    s1 = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', name)
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

def normalize_columns(df):
    """Rename all columns to snake_case."""
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))
    return df

def bronze_transform(df):
    """Full bronze transform: snake_case columns + audit timestamp."""
    return normalize_columns(df).withColumn('_processing_timestamp', current_timestamp())

print('Bronze transforms defined.')

Test the transforms against the in-memory order preview:

In [ ]:
test_df = bronze_transform(spark.sql('SELECT * FROM order_preview'))
display(test_df.limit(3))

### Write the Bronze Stream

Now let's wire up the full bronze stream for `shipment_scan_event`. We read from the archived JSON files (same `_meta` + `data[]` envelope as the Eventstream), apply the bronze transform via `foreachBatch`, and write to [Delta Lake](https://learn.microsoft.com/fabric/data-engineering/lakehouse-and-delta-tables).

Execute the cell below:

In [ ]:
sse_schema = spark.read.json(
    'Files/archive/shipment_scan_event',
    multiLine=True,
    recursiveFileLookup=True
).schema

def write_bronze_sse(batch_df, batch_id):
    transformed = bronze_transform(batch_df)
    transformed.write.format('delta').mode('append').saveAsTable('explore.bronze_shipment_scan_event')

sse_stream_df = spark.readStream.schema(sse_schema).json(
    'Files/archive/shipment_scan_event',
    multiLine=True,
    recursiveFileLookup=True
)

bronze_sse_query = (sse_stream_df.writeStream
    .foreachBatch(write_bronze_sse)
    .option('checkpointLocation', 'Files/explore/checkpoints/bronze_sse')
    .trigger(availableNow=True)
    .start()
)

bronze_sse_query.awaitTermination()
print(f'Bronze SSE rows: {spark.table("explore.bronze_shipment_scan_event").count()}')

Verify what landed in the bronze table — you should see `_meta` (struct), `data` (array), and the new `_processing_timestamp`:

In [ ]:
display(spark.sql('SELECT * FROM explore.bronze_shipment_scan_event LIMIT 5'))

### 🎯 Challenge: Write a Bronze Stream for Orders

Now it's your turn! Wire up a streaming query that:
1. Reads `Files/archive/order` as a JSON stream (use `order_file_schema`)
2. Applies `bronze_transform` via `foreachBatch`
3. Writes to Delta table `explore.bronze_order`

**💡 Hints:**
- The pattern is identical to the SSE bronze stream above
- Use `order_file_schema` (already defined)
- Don't forget a unique `checkpointLocation`

Try it in the cell below!

In [ ]:
# Your code here


<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal the answer</summary>

<br/>

**Approach:** Same `foreachBatch` pattern — swap in the order schema, path, and table name:

```python
def write_bronze_order(batch_df, batch_id):
    transformed = bronze_transform(batch_df)
    transformed.write.format('delta').mode('append').saveAsTable('explore.bronze_order')

order_bronze_df = spark.readStream.schema(order_file_schema).json(
    'Files/archive/order', multiLine=True, recursiveFileLookup=True
)

bronze_order_query = (order_bronze_df.writeStream
    .foreachBatch(write_bronze_order)
    .option('checkpointLocation', 'Files/explore/checkpoints/bronze_order')
    .trigger(availableNow=True)
    .start()
)
bronze_order_query.awaitTermination()
print(f'Bronze Order rows: {spark.table("explore.bronze_order").count()}')
```

**Key Takeaway:** Every bronze stream follows the same three-step recipe: read → transform → write. The only things that change are the schema, path, and table name.

</details>

---

## 5 — Build a Bronze → Silver Streaming Pipeline

The silver layer takes raw bronze data and applies business logic:

| Step | What it does |
|------|--------------|
| **Explode** | The `data` column is an array of records — each element becomes its own row |
| **Flatten** | Nested structs are expanded into top-level columns |
| **Audit columns** | Carry `_processing_timestamp` forward from bronze |

Let's look at the bronze data first — notice the `_meta` struct and the `data` array:

In [ ]:
display(spark.sql('SELECT * FROM explore.bronze_shipment_scan_event LIMIT 3'))

### Develop the Silver Transform

Run the cell below to define the silver transform and test it with a batch read from the bronze table:

In [ ]:
from pyspark.sql.functions import explode, col

def silver_sse_transform(df):
    """Explode the data array and flatten nested structs."""
    exploded = df.select(
        col('_meta'),
        explode('data').alias('record'),
        col('_processing_timestamp')
    )
    flattened = exploded.select(
        col('_meta.*'),
        col('record.*'),
        col('_processing_timestamp')
    )
    return normalize_columns(flattened)

# Test with a batch read — fast feedback
bronze_df = spark.table('explore.bronze_shipment_scan_event')
silver_preview = silver_sse_transform(bronze_df)
display(silver_preview.limit(10))

### Write the Silver Stream

Wire it up as a stream — reading incrementally from the bronze [Delta table as a streaming source](https://learn.microsoft.com/fabric/data-engineering/lakehouse-streaming):

In [ ]:
def write_silver_sse(batch_df, batch_id):
    transformed = silver_sse_transform(batch_df)
    transformed.write.format('delta').mode('append').saveAsTable('explore.silver_shipment_scan_event')

silver_stream_df = spark.readStream.table('explore.bronze_shipment_scan_event')

silver_query = (silver_stream_df.writeStream
    .foreachBatch(write_silver_sse)
    .option('checkpointLocation', 'Files/explore/checkpoints/silver_sse')
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()
print(f'Silver SSE rows: {spark.table("explore.silver_shipment_scan_event").count()}')

The nested `data[]` array has been exploded into individual rows with all fields at the top level. Let's verify:

In [ ]:
display(spark.sql('SELECT * FROM explore.silver_shipment_scan_event LIMIT 20'))

### 🎯 Challenge: Explore the Silver Data with SQL

Write a query that counts the number of scan events **by event type** in the silver table. Order the results by count descending.

**💡 Hints:**
- The column is `event_type` (snake_case after normalization)
- Use `GROUP BY` and `ORDER BY`

In [ ]:
%%sql


<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal the answer</summary>

<br/>

```sql
SELECT event_type, COUNT(*) AS event_count
FROM explore.silver_shipment_scan_event
GROUP BY event_type
ORDER BY event_count DESC;
```

**What to notice:** The event types map to the shipment lifecycle — `Created`, `Received`, `Processed`, `Departed`, `InTransit`, `OutForDelivery`, `Delivered`, plus exception events. This distribution tells you how far most shipments have progressed.

</details>

### 🎯 Challenge: Write a Silver Transform for Shipments

The raw shipment JSON also uses the `_meta` + `data[]` envelope. Each record in `data` is a flat shipment object (no further nesting to explode).

Write a `silver_shipment_transform` function and test it against `explore.shipment_raw` (the table you created in Section 2).

**💡 Hints:**
- Explode `data[]` into individual rows
- Select the struct fields with `.*`
- Normalize column names to `snake_case` using `normalize_columns`

In [ ]:
# Your code here


<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal the answer</summary>

<br/>

**Approach:** Same explode → expand pattern — but the shipment records are flat so there's no second level of nesting:

```python
def silver_shipment_transform(df):
    exploded = df.select(
        explode('data').alias('shipment')
    )
    return normalize_columns(
        exploded.select('shipment.*')
    )

silver_ship_preview = silver_shipment_transform(spark.table('explore.shipment_raw'))
print(f'Columns: {silver_ship_preview.columns}')
display(silver_ship_preview.limit(10))
```

**Key Takeaway:** The bronze and silver patterns are composable. Once you learn the explode → flatten recipe, you can apply it to any entity regardless of nesting depth.

</details>

---

## 🏆 Key Takeaways

| Concept | What you practiced |
|---------|-----------|
| **`availableNow` trigger** | Batch scheduling with streaming state — process only new data each run |
| **Checkpoint state** | Re-running a stream skips already-processed files automatically |
| **`memory` sink** | Iterate on transforms without touching Delta — the safe dev loop |
| **`foreachBatch`** | Full DataFrame API per micro-batch — write to multiple sinks or apply complex logic |
| **`lastProgress` / `status`** | Monitor any running or completed stream interactively |
| **Eventstream (Kafka) pattern** | Same transforms whether the source is files or a message broker |
| **Bronze pattern** | Schema enforcement + column normalization + audit timestamp |
| **Silver pattern** | Explode arrays, flatten structs, carry audit metadata forward |

### 📚 Further Reading

- [Spark Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)
- [Fabric Lakehouse Streaming](https://learn.microsoft.com/fabric/data-engineering/lakehouse-streaming)
- [Delta Lake Table Streaming Reads and Writes](https://docs.delta.io/latest/delta-streaming.html)

> **Up next → Part 2:** We take the exact same transforms and package them into a reusable streaming framework (ArcFlow) — see how software engineering practices change everything.